# Notebook 3: Stationarity Testing + Ornstein-Uhlenbeck Fit

Validates that the HYSYS-weighted refinery margin is mean-reverting,
then fits an Ornstein-Uhlenbeck process to extract the key parameters
that drive the trading signal.

## Tests performed
1. **ADF (Augmented Dickey-Fuller)** — rejects unit root (H₀: non-stationary)
2. **KPSS** — confirms stationarity (H₀: stationary)
3. **OU parameter estimation** — Maximum Likelihood Estimation

## OU Process
```
dX(t) = θ(μ − X(t))dt + σdW(t)
```
- **θ (theta):** mean reversion speed
- **μ (mu):** long-run equilibrium margin
- **σ (sigma):** volatility
- **Half-life:** ln(2)/θ — practical trading horizon in days

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load spread data
df = pd.read_csv('../data/spread_data.csv', index_col=0, parse_dates=True)
margin = df['margin_hysys'].dropna()
print(f'Series length: {len(margin)} observations')
print(f'Mean: ${margin.mean():.2f}/bbl')
print(f'Std:  ${margin.std():.2f}/bbl')

In [ ]:
# ============================================================
# STATIONARITY TESTS
# ============================================================

# ADF Test — H0: unit root (non-stationary)
adf_stat, adf_pvalue, adf_lags, adf_nobs, adf_crit, _ = adfuller(margin, autolag='AIC')

print('=' * 55)
print('AUGMENTED DICKEY-FULLER TEST')
print('H0: Unit root exists (series is NON-stationary)')
print('=' * 55)
print(f'ADF Statistic:  {adf_stat:.4f}')
print(f'p-value:        {adf_pvalue:.4f}')
print(f'Lags used:      {adf_lags}')
print('Critical values:')
for key, val in adf_crit.items():
    print(f'  {key}: {val:.4f}')
print()
if adf_pvalue < 0.05:
    print('RESULT: Reject H0 at 5% significance → series IS stationary ✓')
else:
    print('RESULT: Fail to reject H0 → series may NOT be stationary ✗')

In [ ]:
# KPSS Test — H0: series IS stationary
kpss_stat, kpss_pvalue, kpss_lags, kpss_crit = kpss(margin, regression='c', nlags='auto')

print('=' * 55)
print('KPSS TEST')
print('H0: Series IS stationary')
print('=' * 55)
print(f'KPSS Statistic: {kpss_stat:.4f}')
print(f'p-value:        {kpss_pvalue:.4f}')
print('Critical values:')
for key, val in kpss_crit.items():
    print(f'  {key}: {val:.4f}')
print()
if kpss_pvalue > 0.05:
    print('RESULT: Fail to reject H0 → series IS stationary ✓')
else:
    print('RESULT: Reject H0 → series may NOT be stationary ✗')

In [ ]:
# ============================================================
# ORNSTEIN-UHLENBECK PARAMETER ESTIMATION (MLE)
# ============================================================

def ou_neg_log_likelihood(params, X, dt=1.0):
    theta, mu, sigma = params
    if theta <= 0 or sigma <= 0:
        return 1e10
    n = len(X) - 1
    X_t  = X[:-1]
    X_t1 = X[1:]
    exp_decay = np.exp(-theta * dt)
    exp_val   = X_t * exp_decay + mu * (1 - exp_decay)
    var       = (sigma**2 / (2 * theta)) * (1 - np.exp(-2 * theta * dt))
    if var <= 0:
        return 1e10
    ll = (-n / 2) * np.log(2 * np.pi * var) \
         - np.sum((X_t1 - exp_val)**2) / (2 * var)
    return -ll

X = margin.values
x0 = [0.1, X.mean(), X.std()]

result = minimize(
    ou_neg_log_likelihood,
    x0=x0,
    args=(X,),
    method='L-BFGS-B',
    bounds=[(1e-6, 10), (None, None), (1e-6, None)]
)

theta, mu, sigma = result.x
half_life = np.log(2) / theta

print('=' * 55)
print('ORNSTEIN-UHLENBECK PARAMETERS (MLE)')
print('dX = θ(μ − X)dt + σdW')
print('=' * 55)
print(f'θ (mean reversion speed): {theta:.4f} per day')
print(f'μ (long-run mean):        ${mu:.4f}/bbl')
print(f'σ (volatility):           ${sigma:.4f}/bbl/day^0.5')
print(f'Half-life:                {half_life:.1f} trading days')
print(f'Optimisation success:     {result.success}')

In [ ]:
# ============================================================
# PLOT: Margin series with mean and ±1σ, ±2σ bands
# ============================================================

# Rolling z-score (252-day lookback)
lookback = 252
df['margin_mean'] = df['margin_hysys'].rolling(lookback).mean()
df['margin_std']  = df['margin_hysys'].rolling(lookback).std()
df['zscore']      = (df['margin_hysys'] - df['margin_mean']) / df['margin_std']

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# Margin with bands
axes[0].plot(df.index, df['margin_hysys'], color='#2c3e50', linewidth=0.8, alpha=0.8, label='HYSYS margin')
axes[0].plot(df.index, df['margin_mean'], color='#e74c3c', linewidth=1.5, label='Rolling mean (252d)')
axes[0].fill_between(df.index,
    df['margin_mean'] - df['margin_std'],
    df['margin_mean'] + df['margin_std'],
    alpha=0.15, color='#e74c3c', label='±1σ band')
axes[0].fill_between(df.index,
    df['margin_mean'] - 2*df['margin_std'],
    df['margin_mean'] + 2*df['margin_std'],
    alpha=0.08, color='#e74c3c', label='±2σ band')
axes[0].set_ylabel('Margin ($/bbl)')
axes[0].set_title(f'HYSYS Refinery Margin — OU Half-life: {half_life:.0f} trading days')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Z-score
axes[1].plot(df.index, df['zscore'], color='#2c3e50', linewidth=0.8)
axes[1].axhline(0,    color='#e74c3c', linewidth=1, linestyle='-')
axes[1].axhline(1.5,  color='#e67e22', linewidth=1, linestyle='--', label='±1.5σ entry threshold')
axes[1].axhline(-1.5, color='#e67e22', linewidth=1, linestyle='--')
axes[1].fill_between(df.index, df['zscore'], 0,
    where=(df['zscore'] < -1.5), alpha=0.3, color='#27ae60', label='Long signal zone')
axes[1].fill_between(df.index, df['zscore'], 0,
    where=(df['zscore'] > 1.5),  alpha=0.3, color='#e74c3c', label='Short signal zone')
axes[1].set_ylabel('Z-score')
axes[1].set_xlabel('Date')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/stationarity_ou_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save OU parameters and z-score series for notebook 4
ou_params = pd.Series({'theta': theta, 'mu': mu, 'sigma': sigma, 'half_life': half_life})
ou_params.to_csv('../data/ou_parameters.csv', header=['value'])

df[['margin_hysys', 'crack_321', 'margin_mean', 'margin_std', 'zscore']].to_csv('../data/spread_with_zscore.csv')
print('Saved OU parameters and z-score series.')
print(ou_params)